# Notebook 17 — Hierarchical Prototype Routing

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 16 managed a prototype memory bank with aging and reinforcement.

Notebook 17 turns the memory bank into a routing hierarchy:

- parent prototypes,
- child prototypes,
- coarse-to-fine routing,
- merge candidates,
- split candidates,
- hierarchical route stability.

Constraint view:
> adaptive runtimes should route from broad structure to precise execution behavior.

## Goals

1. Load Notebook 16 memory-bank outputs when available.
2. Build feature vectors for prototypes.
3. Cluster prototypes into parent groups.
4. Assign windows through a coarse-to-fine route:
   - parent route
   - child prototype route
   - policy route
5. Detect merge candidates and split candidates.
6. Score hierarchical routing stability.
7. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load memory-bank outputs

Uses Notebook 16 outputs if available.

In [ ]:
events_path = RESULTS_DIR / "notebook16_prototype_memory_bank_aging.csv"
memory_path = RESULTS_DIR / "notebook16_memory_bank_summary.csv"
proto_path = RESULTS_DIR / "notebook15_updated_prototypes.csv"

if events_path.exists():
    events = pd.read_csv(events_path)
    print("Loaded:", events_path)
else:
    events = None

if memory_path.exists():
    memory = pd.read_csv(memory_path)
    print("Loaded:", memory_path)
else:
    memory = None

if proto_path.exists():
    prototypes = pd.read_csv(proto_path)
    print("Loaded:", proto_path)
else:
    prototypes = None

if events is None or memory is None or prototypes is None:
    print("Missing prior outputs; creating fallback hierarchy data.")
    rng = np.random.default_rng(42)
    proto_names = [
        "low_entropy_repeating",
        "sequential_ids",
        "uniform_32bit",
        "zipfian_smallints",
        "clustered_ranges",
        "learned_drift_prototype",
    ]
    prototypes = pd.DataFrame({
        "regime": proto_names,
        "entropy_norm": [0.10, 0.90, 1.00, 0.18, 0.55, 0.51],
        "repetition_ratio": [0.98, 0.00, 0.00, 0.80, 0.98, 0.46],
        "locality_small_delta_ratio": [1.00, 1.00, 0.00, 0.27, 0.00, 0.44],
        "cache_window_reuse_proxy": [0.94, 0.00, 0.00, 0.35, 0.01, 0.41],
        "branch_norm": [0.02, 0.20, 0.72, 0.72, 0.79, 0.64],
        "coherence_score": [0.95, 0.55, 0.08, 0.42, 0.18, 0.45],
        "hardware_pressure_proxy": [0.02, 0.25, 0.88, 0.72, 0.98, 0.69],
    })
    memory = pd.DataFrame({
        "prototype": proto_names,
        "memory_stability_score": [0.56, 0.38, 0.58, 0.49, 0.42, 0.60],
        "effective_memory_weight": [0.62, 0.48, 0.66, 0.57, 0.50, 0.56],
        "memory_action": ["retain", "refresh", "retain", "refresh", "refresh", "retain"],
        "usage_count": [32, 22, 36, 26, 20, 64],
    })
    n = 220
    rows = []
    for i in range(n):
        if 88 <= i <= 145:
            dom = "learned_drift_prototype"
        else:
            dom = rng.choice(proto_names[:-1])
        rows.append({
            "window_id": i,
            "new_dominant_prototype": dom,
            "updated_policy": {
                "low_entropy_repeating": "coherent_local",
                "sequential_ids": "hybrid",
                "uniform_32bit": "simd",
                "zipfian_smallints": "hybrid",
                "clustered_ranges": "guarded_fallback",
                "learned_drift_prototype": "prototype_recovery",
            }[dom],
            "new_residual": abs(rng.normal(0.25, 0.06)),
            "new_drift_score": abs(rng.normal(0.12, 0.05)),
        })
    events = pd.DataFrame(rows)

events.head(), memory.head(), prototypes.head()

## Normalize prototype feature table

In [ ]:
feature_cols = [
    "entropy_norm",
    "repetition_ratio",
    "locality_small_delta_ratio",
    "cache_window_reuse_proxy",
    "branch_norm",
    "coherence_score",
    "hardware_pressure_proxy",
]

protos = prototypes.copy()
if "regime" not in protos.columns:
    protos["regime"] = [f"prototype_{i}" for i in range(len(protos))]

for c in feature_cols:
    if c not in protos.columns:
        protos[c] = 0.5
    protos[c] = pd.to_numeric(protos[c], errors="coerce").fillna(0.5)

mem = memory.copy()
if "prototype" not in mem.columns:
    mem["prototype"] = protos["regime"]

protos = protos.merge(mem, left_on="regime", right_on="prototype", how="left")
for c in ["memory_stability_score", "effective_memory_weight", "usage_count"]:
    if c not in protos.columns:
        protos[c] = 0.0
    protos[c] = pd.to_numeric(protos[c], errors="coerce").fillna(0.0)

protos[["regime"] + feature_cols + ["memory_stability_score", "effective_memory_weight", "usage_count"]]

## Build hierarchy: parent prototype clusters

With small prototype banks, use 3 parent clusters by default.

In [ ]:
X = protos[feature_cols].to_numpy(float)
n_clusters = min(3, max(1, len(protos)))

if len(protos) > 1:
    clusterer = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
    parent_ids = clusterer.fit_predict(X)
else:
    parent_ids = np.array([0])

protos["parent_id"] = parent_ids
protos["parent_route"] = ["parent_" + str(i) for i in protos["parent_id"]]

parent_summary = (
    protos.groupby("parent_route", as_index=False)
    .agg(
        prototype_count=("regime", "size"),
        mean_memory_stability=("memory_stability_score", "mean"),
        mean_effective_weight=("effective_memory_weight", "mean"),
        total_usage=("usage_count", "sum"),
        mean_entropy=("entropy_norm", "mean"),
        mean_pressure=("hardware_pressure_proxy", "mean"),
        mean_coherence=("coherence_score", "mean"),
    )
)

protos[["regime", "parent_route"]], parent_summary

## Coarse-to-fine route assignment for windows

Each window now has:

```text
parent route → child prototype → execution policy
```

In [ ]:
route_map = protos.set_index("regime")["parent_route"].to_dict()

work = events.copy().sort_values("window_id").reset_index(drop=True)

if "new_dominant_prototype" not in work.columns:
    work["new_dominant_prototype"] = "unknown"
if "updated_policy" not in work.columns:
    work["updated_policy"] = "unknown"

work["parent_route"] = work["new_dominant_prototype"].map(route_map).fillna("parent_unknown")
work["child_route"] = work["new_dominant_prototype"]
work["hierarchical_route"] = (
    work["parent_route"].astype(str)
    + " / "
    + work["child_route"].astype(str)
    + " / "
    + work["updated_policy"].astype(str)
)

work["parent_changed"] = work["parent_route"].ne(work["parent_route"].shift(1)).fillna(False)
work["child_changed"] = work["child_route"].ne(work["child_route"].shift(1)).fillna(False)
work["policy_changed"] = work["updated_policy"].ne(work["updated_policy"].shift(1)).fillna(False)

window = 15
work["parent_switch_rate"] = work["parent_changed"].rolling(window, min_periods=1).mean()
work["child_switch_rate"] = work["child_changed"].rolling(window, min_periods=1).mean()
work["policy_switch_rate"] = work["policy_changed"].rolling(window, min_periods=1).mean()

work[["window_id", "parent_route", "child_route", "updated_policy", "hierarchical_route"]].head()

## Route stability score

Hierarchical stability rewards stable parent routes even when child routes vary.

In [ ]:
work["hierarchical_stability_score"] = (
    1.0
    - 0.50 * work["parent_switch_rate"]
    - 0.30 * work["child_switch_rate"]
    - 0.20 * work["policy_switch_rate"]
).clip(0, 1)

work[["window_id", "parent_switch_rate", "child_switch_rate", "policy_switch_rate", "hierarchical_stability_score"]].head()

## Detect merge and split candidates

- **Merge candidates:** prototypes in same parent cluster with low feature distance.
- **Split candidates:** prototypes with low stability or high usage but high internal switching.

In [ ]:
D = pairwise_distances(X)
merge_rows = []
for i in range(len(protos)):
    for j in range(i + 1, len(protos)):
        same_parent = protos.loc[i, "parent_route"] == protos.loc[j, "parent_route"]
        if same_parent:
            merge_rows.append({
                "prototype_a": protos.loc[i, "regime"],
                "prototype_b": protos.loc[j, "regime"],
                "parent_route": protos.loc[i, "parent_route"],
                "feature_distance": float(D[i, j]),
                "merge_candidate": bool(D[i, j] < np.quantile(D[D > 0], 0.35)) if np.any(D > 0) else False,
            })

merge_candidates = pd.DataFrame(merge_rows)

child_switch_by_proto = (
    work.groupby("child_route", as_index=False)
    .agg(
        windows=("window_id", "size"),
        mean_child_switch=("child_switch_rate", "mean"),
        mean_parent_switch=("parent_switch_rate", "mean"),
        mean_stability=("hierarchical_stability_score", "mean"),
    )
    .rename(columns={"child_route": "prototype"})
)

split_candidates = protos[["regime", "parent_route", "memory_stability_score", "usage_count"]].rename(columns={"regime": "prototype"})
split_candidates = split_candidates.merge(child_switch_by_proto, on="prototype", how="left")
for c in ["windows", "mean_child_switch", "mean_parent_switch", "mean_stability"]:
    split_candidates[c] = split_candidates[c].fillna(0)

split_candidates["split_candidate"] = (
    (split_candidates["usage_count"] > split_candidates["usage_count"].median()) &
    (split_candidates["mean_stability"] < split_candidates["mean_stability"].median())
) | (
    split_candidates["mean_child_switch"] > split_candidates["mean_child_switch"].quantile(0.75)
)

merge_candidates, split_candidates

## Export hierarchical routing tables

In [ ]:
csv_path = RESULTS_DIR / "notebook17_hierarchical_prototype_routing.csv"
json_path = RESULTS_DIR / "notebook17_hierarchical_prototype_routing.json"
proto_csv_path = RESULTS_DIR / "notebook17_prototype_hierarchy.csv"
parent_csv_path = RESULTS_DIR / "notebook17_parent_route_summary.csv"
merge_csv_path = RESULTS_DIR / "notebook17_merge_candidates.csv"
split_csv_path = RESULTS_DIR / "notebook17_split_candidates.csv"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)
protos.to_csv(proto_csv_path, index=False)
parent_summary.to_csv(parent_csv_path, index=False)
merge_candidates.to_csv(merge_csv_path, index=False)
split_candidates.to_csv(split_csv_path, index=False)

print("Saved:", csv_path)
print("Saved:", json_path)
print("Saved:", proto_csv_path)
print("Saved:", parent_csv_path)
print("Saved:", merge_csv_path)
print("Saved:", split_csv_path)

## Figure 1 — Prototype hierarchy projection

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook17_prototype_hierarchy_projection.png"

if len(protos) >= 2:
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X)
else:
    coords = np.zeros((len(protos), 2))

protos["pca_x"] = coords[:, 0]
protos["pca_y"] = coords[:, 1]

plt.figure(figsize=(8, 6))
for parent in sorted(protos["parent_route"].unique()):
    part = protos[protos["parent_route"] == parent]
    plt.scatter(part["pca_x"], part["pca_y"], s=90, label=parent)
    for _, r in part.iterrows():
        plt.text(r["pca_x"], r["pca_y"], r["regime"], fontsize=8)
plt.xlabel("Prototype PC1")
plt.ylabel("Prototype PC2")
plt.title("Hierarchical Prototype Routing: Parent Clusters")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Parent route timeline

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook17_parent_route_timeline.png"

labels = sorted(work["parent_route"].unique())
lab_to_id = {lab: i for i, lab in enumerate(labels)}

plt.figure(figsize=(12, 4))
plt.step(work["window_id"], work["parent_route"].map(lab_to_id), where="mid")
plt.yticks(list(lab_to_id.values()), list(lab_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Parent route")
plt.title("Hierarchical Routing: Parent Route Timeline")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Child prototype timeline

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook17_child_prototype_timeline.png"

labels = sorted(work["child_route"].unique())
lab_to_id = {lab: i for i, lab in enumerate(labels)}

plt.figure(figsize=(12, 5))
plt.step(work["window_id"], work["child_route"].map(lab_to_id), where="mid")
plt.yticks(list(lab_to_id.values()), list(lab_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Child prototype")
plt.title("Hierarchical Routing: Child Prototype Timeline")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Route switch rates

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook17_route_switch_rates.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["parent_switch_rate"], label="parent switch rate")
plt.plot(work["window_id"], work["child_switch_rate"], label="child switch rate")
plt.plot(work["window_id"], work["policy_switch_rate"], label="policy switch rate")
plt.xlabel("Window")
plt.ylabel("Rolling switch rate")
plt.title("Hierarchical Routing: Switch Rates")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Hierarchical stability timeline

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook17_hierarchical_stability_timeline.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["hierarchical_stability_score"])
plt.xlabel("Window")
plt.ylabel("Hierarchical stability score")
plt.title("Hierarchical Routing: Stability Over Time")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Parent route summary matrix

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook17_parent_route_summary_matrix.png"

mat_cols = [
    "prototype_count",
    "mean_memory_stability",
    "mean_effective_weight",
    "total_usage",
    "mean_entropy",
    "mean_pressure",
    "mean_coherence",
]

mat = parent_summary.set_index("parent_route")[mat_cols].copy()
for c in mat.columns:
    lo, hi = mat[c].min(), mat[c].max()
    if hi != lo:
        mat[c] = (mat[c] - lo) / (hi - lo)
    else:
        mat[c] = 1.0

plt.figure(figsize=(10, 4))
plt.imshow(mat.values, aspect="auto", vmin=0, vmax=1)
plt.yticks(range(len(mat.index)), mat.index)
plt.xticks(range(len(mat.columns)), mat.columns, rotation=45, ha="right")
plt.colorbar(label="Normalized score")
plt.title("Hierarchical Routing: Parent Route Summary Matrix")
plt.tight_layout()
plt.savefig(fig_path_6, dpi=160)
plt.show()

print("Saved:", fig_path_6)

## Figure 7 — Merge/split candidate counts

In [ ]:
fig_path_7 = FIGURES_DIR / "notebook17_merge_split_counts.png"

counts = pd.DataFrame([
    {"category": "merge candidates", "count": int(merge_candidates["merge_candidate"].sum()) if len(merge_candidates) else 0},
    {"category": "split candidates", "count": int(split_candidates["split_candidate"].sum()) if len(split_candidates) else 0},
])

plt.figure(figsize=(7, 4))
plt.bar(counts["category"], counts["count"])
plt.ylabel("Count")
plt.title("Hierarchical Routing: Merge / Split Candidates")
plt.tight_layout()
plt.savefig(fig_path_7, dpi=160)
plt.show()

print("Saved:", fig_path_7)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_17_hierarchical_prototype_routing.md"

summary = {
    "windows": int(len(work)),
    "prototype_count": int(len(protos)),
    "parent_route_count": int(protos["parent_route"].nunique()),
    "mean_hierarchical_stability": float(work["hierarchical_stability_score"].mean()),
    "mean_parent_switch_rate": float(work["parent_switch_rate"].mean()),
    "mean_child_switch_rate": float(work["child_switch_rate"].mean()),
    "mean_policy_switch_rate": float(work["policy_switch_rate"].mean()),
    "merge_candidate_count": int(merge_candidates["merge_candidate"].sum()) if len(merge_candidates) else 0,
    "split_candidate_count": int(split_candidates["split_candidate"].sum()) if len(split_candidates) else 0,
}

lines = [
    "# Report 17 — Hierarchical Prototype Routing",
    "",
    "This report turns the prototype memory bank into a coarse-to-fine routing hierarchy.",
    "",
    "Constraint view:",
    "> adaptive runtimes should route from broad structure to precise execution behavior.",
    "",
    "## Generated outputs",
    "",
    f"- Window routes CSV: `{csv_path}`",
    f"- Window routes JSON: `{json_path}`",
    f"- Prototype hierarchy CSV: `{proto_csv_path}`",
    f"- Parent route summary CSV: `{parent_csv_path}`",
    f"- Merge candidates CSV: `{merge_csv_path}`",
    f"- Split candidates CSV: `{split_csv_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    f"- Figure: `{fig_path_6}`",
    f"- Figure: `{fig_path_7}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Parent route summary",
    "",
    parent_summary.to_markdown(index=False),
    "",
    "## Merge candidates",
    "",
    merge_candidates.to_markdown(index=False) if len(merge_candidates) else "No merge candidates.",
    "",
    "## Split candidates",
    "",
    split_candidates.to_markdown(index=False),
    "",
    "## Interpretation",
    "",
    "- Parent routes provide stable coarse structure even when child prototypes switch.",
    "- Child routes preserve precision for execution-policy choice.",
    "- Merge candidates identify redundant prototypes within a parent group.",
    "- Split candidates identify broad or unstable prototypes that may need refinement.",
    "",
    "## Next step",
    "",
    "Notebook 18 can perform prototype compression: merge redundant prototypes while preserving reconstruction quality.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook17_hierarchical_prototype_routing_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook17_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_17_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))